In [1]:
!pip install -q sentence-transformers


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('Model loaded successfully')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully


In [3]:
sentences = [
    'The student is learning machine learning.',
    'The learner is studying artificial intelligence.',
    'A train arrived at the railway station.',
    'Fresh vegetables are available in the market.'
]

for i, sentence in enumerate(sentences, start=1):
    print(f'{i}. {sentence}')


1. The student is learning machine learning.
2. The learner is studying artificial intelligence.
3. A train arrived at the railway station.
4. Fresh vegetables are available in the market.


In [4]:
embeddings = model.encode(sentences)

print('Number of sentences:', len(embeddings))
print('Embedding shape:', embeddings.shape)
print('First embedding (first 10 values):')
print(embeddings[0][:10])


Number of sentences: 4
Embedding shape: (4, 384)
First embedding (first 10 values):
[-0.00037504  0.03260435  0.00616778 -0.05656463  0.01627318 -0.02873934
  0.02182608 -0.00550216 -0.07055108  0.00268341]


In [5]:
print('Embedding dimension:', len(embeddings[0]))
print('Vector type:', type(embeddings[0]))


Embedding dimension: 384
Vector type: <class 'numpy.ndarray'>


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)

print('Cosine similarity matrix:')
print(similarity_matrix)


Cosine similarity matrix:
[[ 0.9999999   0.70373756  0.10047738  0.00590549]
 [ 0.70373756  0.9999997   0.08160008 -0.00983799]
 [ 0.10047738  0.08160008  1.          0.0017435 ]
 [ 0.00590549 -0.00983799  0.0017435   1.0000002 ]]


In [7]:
pairs = [
    (0, 1),  # learning / studying
    (0, 2),  # learning / train
    (2, 3)   # train / vegetables
]

for a, b in pairs:
    score = cosine_similarity([embeddings[a]], [embeddings[b]])[0][0]
    print(f'Pair: {a+1} and {b+1}')
    print(f'Similarity: {score:.4f}\n')


Pair: 1 and 2
Similarity: 0.7037

Pair: 1 and 3
Similarity: 0.1005

Pair: 3 and 4
Similarity: 0.0017



In [8]:
query = 'I am studying artificial intelligence and machine learning.'
query_embedding = model.encode([query])

query_scores = cosine_similarity(query_embedding, embeddings)[0]

for sentence, score in sorted(zip(sentences, query_scores), key=lambda x: x[1], reverse=True):
    print(f'{score:.4f}  ->  {sentence}')


0.5635  ->  The learner is studying artificial intelligence.
0.4953  ->  The student is learning machine learning.
0.0381  ->  Fresh vegetables are available in the market.
-0.0202  ->  A train arrived at the railway station.


In [9]:
best_index = query_scores.argmax()
print('Query:', query)
print('Most similar sentence:', sentences[best_index])
print('Similarity score:', round(float(query_scores[best_index]), 4))


Query: I am studying artificial intelligence and machine learning.
Most similar sentence: The learner is studying artificial intelligence.
Similarity score: 0.5635
